# Datasets & Synthetic Test Generation

This notebook covers how to create, manage, and automatically generate evaluation datasets using both **DeepEval** and **RAGAS**.

### What We Cover
1. **DeepEval EvaluationDataset** — Create from scratch, JSON, CSV
2. **DeepEval Synthesizer** — Auto-generate test cases from documents
3. **Pytest integration** — Run evaluations in CI/CD
4. **RAGAS TestsetGenerator** — Generate test sets from knowledge base
5. **Comparison** — DeepEval Synthesizer vs RAGAS TestsetGenerator

---
## 1. Setup

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv

dotenv_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(dotenv_path)

from openai import OpenAI
from deepeval.test_case import LLMTestCase
from deepeval.dataset import EvaluationDataset
from deepeval import evaluate as deepeval_evaluate

# RAGAS imports
from ragas.testset import TestsetGenerator
from ragas import evaluate as ragas_evaluate, EvaluationDataset as RagasEvaluationDataset, SingleTurnSample
from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextRecall
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document as LCDocument

# Load pipeline results
data_path = os.path.join(os.getcwd(), "data", "pipeline_results.json")
with open(data_path) as f:
    pipeline_results = json.load(f)

# Create test cases
test_cases = []
for r in pipeline_results:
    tc = LLMTestCase(
        input=r["query"],
        actual_output=r["actual_output"],
        expected_output=r.get("expected_output", ""),
        retrieval_context=r["retrieval_context"],
    )
    test_cases.append(tc)

# Acme Corp knowledge base (used by RAGAS TestsetGenerator later)
ACME_DOCUMENTS = [
    {"id": 1, "title": "Return Policy Overview", "content": (
        "Acme Corp offers a 30-day return policy on all products purchased through "
        "our website or retail stores. Items must be unused, in original packaging, "
        "with receipt. Refunds are processed within 5-7 business days. Shipping costs "
        "for returns are the customer's responsibility unless the item was defective."
    )},
    {"id": 2, "title": "Electronics Return Policy", "content": (
        "Electronics have a 15-day return window. All electronics must be returned with "
        "original accessories and packaging. A 15% restocking fee may apply to opened "
        "electronics. Defective electronics can be exchanged within 90 days."
    )},
    {"id": 3, "title": "Shipping Policy", "content": (
        "Acme Corp offers Standard Shipping (5-7 days, free over $50), Expedited Shipping "
        "(2-3 days, $12.99), and Overnight Shipping (next business day, $24.99 if ordered "
        "before 2 PM EST). Orders are processed within 1-2 business days."
    )},
    {"id": 4, "title": "Product Warranty Information", "content": (
        "All Acme Corp branded products come with a 1-year limited warranty covering "
        "defects in materials and workmanship. Extended warranty plans (2-year at $29.99 "
        "and 3-year at $49.99) are available at time of purchase."
    )},
    {"id": 5, "title": "Accepted Payment Methods", "content": (
        "Acme Corp accepts Visa, MasterCard, American Express, Discover, PayPal, Apple Pay, "
        "Google Pay, and Acme Gift Cards. For orders over $200, Acme Pay Later splits "
        "payments into 4 interest-free installments over 6 weeks."
    )},
    {"id": 6, "title": "Acme Rewards Loyalty Program", "content": (
        "Acme Rewards is free to join, earning 1 point per dollar spent. 100 points = $5 off. "
        "Silver tier (500+ pts/year): free expedited shipping. Gold tier (1000+ pts/year): "
        "priority support and exclusive product previews. Points expire after 12 months of inactivity."
    )},
    {"id": 7, "title": "Product: Acme SmartHome Hub", "content": (
        "The Acme SmartHome Hub is priced at $149.99. It supports WiFi, Bluetooth, Zigbee, "
        "and Z-Wave protocols. Features include voice control, 5-inch touchscreen, energy "
        "monitoring, and automated routines. Comes with a 2-year warranty."
    )},
]

# Simple RAG pipeline for RAGAS evaluation
openai_client = OpenAI()

def get_embeddings(texts):
    response = openai_client.embeddings.create(input=texts, model="text-embedding-3-small")
    return [item.embedding for item in response.data]

def rag_pipeline(question):
    """Simple RAG pipeline: embed query, find best matching document, generate answer."""
    query_emb = get_embeddings([question])[0]
    
    # Find most relevant documents via cosine similarity
    import numpy as np
    doc_texts = [f"{d['title']}\n\n{d['content']}" for d in ACME_DOCUMENTS]
    doc_embs = get_embeddings(doc_texts)
    
    similarities = [
        np.dot(query_emb, de) / (np.linalg.norm(query_emb) * np.linalg.norm(de))
        for de in doc_embs
    ]
    top_indices = sorted(range(len(similarities)), key=lambda i: similarities[i], reverse=True)[:3]
    contexts = [doc_texts[i] for i in top_indices]
    
    context_str = "\n\n".join(contexts)
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer the question based on the provided context. Be concise."},
            {"role": "user", "content": f"Context: {context_str}\n\nQuestion: {question}"},
        ],
        temperature=0.1,
        max_tokens=200,
    )
    return {"answer": response.choices[0].message.content, "contexts": contexts}

print(f"Loaded {len(test_cases)} test cases.")
print(f"Knowledge base: {len(ACME_DOCUMENTS)} documents.")
print("RAG pipeline defined.")

-
## 5. EvaluationDataset

DeepEval provides an `EvaluationDataset` class to manage collections of test cases. This makes it easy to save, load, and reuse test data.

### 5.1 Create Dataset from Scratch

In [ ]:
# Create a dataset from our existing test cases
dataset = EvaluationDataset()
for tc in test_cases:
    dataset.add_test_case(tc)

print(f"Dataset created with {len(dataset.test_cases)} test cases.")
print(f"\nFirst test case:")
print(f"  Input: {dataset.test_cases[0].input[:60]}...")
print(f"  Output: {dataset.test_cases[0].actual_output[:60]}...")

### 5.2 Load Dataset from JSON

In [ ]:
# First, let's save our test cases as JSON in the format DeepEval expects
output_dir = os.path.join(os.getcwd(), "data")
os.makedirs(output_dir, exist_ok=True)

# Save in a standard format
eval_data = []
for r in pipeline_results:
    eval_data.append({
        "input": r["query"],
        "actual_output": r["actual_output"],
        "expected_output": r["expected_output"],
        "retrieval_context": r["retrieval_context"],
    })

json_path = os.path.join(output_dir, "eval_dataset.json")
with open(json_path, "w") as f:
    json.dump(eval_data, f, indent=2)

print(f"Saved evaluation dataset to {json_path}")

In [ ]:
# Load it back and create test cases
with open(json_path) as f:
    loaded_data = json.load(f)

loaded_test_cases = []
for item in loaded_data:
    tc = LLMTestCase(
        input=item["input"],
        actual_output=item["actual_output"],
        expected_output=item["expected_output"],
        retrieval_context=item["retrieval_context"],
    )
    loaded_test_cases.append(tc)

loaded_dataset = EvaluationDataset()
for tc in loaded_test_cases:
    loaded_dataset.add_test_case(tc)

print(f"Loaded dataset with {len(loaded_dataset.test_cases)} test cases from JSON.")

### 5.3 Load Dataset from CSV

In [ ]:
# Save as CSV
csv_path = os.path.join(output_dir, "eval_dataset.csv")

csv_data = []
for r in pipeline_results:
    csv_data.append({
        "input": r["query"],
        "actual_output": r["actual_output"],
        "expected_output": r["expected_output"],
        "retrieval_context": json.dumps(r["retrieval_context"]),  # JSON-encode the list
    })

csv_df = pd.DataFrame(csv_data)
csv_df.to_csv(csv_path, index=False)
print(f"Saved CSV to {csv_path}")
print(f"\nCSV columns: {list(csv_df.columns)}")
print(csv_df.head(3).to_string(index=False))

In [ ]:
# Load from CSV
loaded_csv = pd.read_csv(csv_path)

csv_test_cases = []
for _, row in loaded_csv.iterrows():
    tc = LLMTestCase(
        input=row["input"],
        actual_output=row["actual_output"],
        expected_output=row["expected_output"],
        retrieval_context=json.loads(row["retrieval_context"]),
    )
    csv_test_cases.append(tc)

csv_dataset = EvaluationDataset()
for tc in csv_test_cases:
    csv_dataset.add_test_case(tc)

print(f"Loaded {len(csv_dataset.test_cases)} test cases from CSV.")

### 5.4 Pytest Integration Pattern

DeepEval integrates with pytest for CI/CD. Here is how you would structure your test file:

```python
# test_rag_pipeline.py
import pytest
from deepeval import assert_test
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric
from deepeval.test_case import LLMTestCase
from deepeval.dataset import EvaluationDataset

# Load your dataset
dataset = EvaluationDataset()
dataset.add_test_cases_from_json_file("data/eval_dataset.json", ...)

# Define metrics
metrics = [
    AnswerRelevancyMetric(threshold=0.7),
    FaithfulnessMetric(threshold=0.7),
]

@pytest.mark.parametrize("test_case", dataset)
def test_rag_quality(test_case: LLMTestCase):
    assert_test(test_case, metrics)
```

Run with: `deepeval test run test_rag_pipeline.py`

In [ ]:
# Simulate the parametrized pattern in a notebook
from deepeval import assert_test
from deepeval.metrics import AnswerRelevancyMetric

metric = AnswerRelevancyMetric(threshold=0.5, model="gpt-4o-mini")

passed = 0
failed = 0

for i, tc in enumerate(test_cases[:5]):
    try:
        assert_test(tc, [metric])
        passed += 1
        score_str = f"{metric.score:.4f}" if metric.score is not None else "None"
        print(f"  Test {i+1}: PASSED (score={score_str})")
    except AssertionError:
        failed += 1
        score_str = f"{metric.score:.4f}" if metric.score is not None else "None"
        print(f"  Test {i+1}: FAILED (score={score_str})")

print(f"\nResults: {passed} passed, {failed} failed out of 5")

-
## 6. Synthetic Data Generation

DeepEval's `Synthesizer` can automatically generate evaluation test cases ("goldens") from your documents. This is useful when you do not have manually labeled test data.

In [ ]:
from deepeval.synthesizer import Synthesizer

print("Synthesizer imported successfully.")

In [ ]:
# Prepare documents for synthesis
# Load knowledge base if available, otherwise use inline
kb_path = os.path.join(os.getcwd(), "data", "knowledge_base.json")

if os.path.exists(kb_path):
    with open(kb_path) as f:
        kb_data = json.load(f)
    doc_texts = [item["text"] for item in kb_data]
else:
    doc_texts = [
        "Acme Corp offers a 30-day return policy on all products. Items must be unused, in original packaging, with proof of purchase. Refunds processed in 5-7 business days.",
        "Electronic products have a 15-day return window. All original accessories must be included. A 15% restocking fee may apply to opened electronics.",
        "Acme Corp offers Standard Shipping (5-7 days, free over $50), Expedited (2-3 days, $12.99), and Overnight ($24.99, next business day).",
        "All Acme Corp products come with a 1-year limited warranty covering defects in materials and workmanship. Extended 2-year and 3-year plans available.",
        "The Acme SmartHome Hub ($149.99) supports WiFi, Bluetooth, Zigbee, Z-Wave. Features: voice control, 5-inch touchscreen, energy monitoring. Compatible with 10,000+ devices.",
    ]

# The Synthesizer API expects file paths, so save each document to a temp file
import tempfile

synth_doc_dir = os.path.join(os.getcwd(), "data", "synth_docs")
os.makedirs(synth_doc_dir, exist_ok=True)

document_paths = []
for i, text in enumerate(doc_texts):
    doc_path = os.path.join(synth_doc_dir, f"doc_{i}.txt")
    with open(doc_path, "w") as f:
        f.write(text)
    document_paths.append(doc_path)

print(f"Saved {len(document_paths)} documents for synthesis to {synth_doc_dir}.")

In [ ]:
# Create synthesizer and generate goldens
synthesizer = Synthesizer(model="gpt-4o-mini")

print("Generating synthetic test cases from documents...")
print("(This may take 1-2 minutes)\n")

# generate_goldens_from_docs returns a list and stores in synthesizer.synthetic_goldens
goldens = synthesizer.generate_goldens_from_docs(
    document_paths=document_paths,
    max_goldens_per_context=2,
)

print(f"\nGenerated {len(goldens)} synthetic test cases (goldens).")

In [ ]:
# Display the generated goldens
for i, golden in enumerate(goldens, 1):
    print(f"Golden {i}:")
    print(f"  Input:    {golden.input[:80]}..." if len(golden.input) > 80 else f"  Input:    {golden.input}")
    print(f"  Expected: {golden.expected_output[:80]}..." if golden.expected_output and len(golden.expected_output) > 80 else f"  Expected: {golden.expected_output}")
    print()

In [ ]:
# Convert goldens to a dataset and evaluate
# First, we need to generate actual outputs using our RAG pipeline (or simulate them)
from openai import OpenAI

client = OpenAI()

synthetic_test_cases = []

for golden in goldens:
    # Generate a response using the LLM (simulating RAG pipeline)
    context_text = golden.context if hasattr(golden, 'context') and golden.context else ""
    if isinstance(context_text, list):
        context_text = " ".join(context_text)
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer the question based on the provided context. Be concise."},
            {"role": "user", "content": f"Context: {context_text}\n\nQuestion: {golden.input}"},
        ],
        temperature=0.1,
        max_tokens=200,
    )
    
    actual_output = response.choices[0].message.content
    
    retrieval_ctx = golden.context if hasattr(golden, 'context') and golden.context else []
    if isinstance(retrieval_ctx, str):
        retrieval_ctx = [retrieval_ctx]
    
    tc = LLMTestCase(
        input=golden.input,
        actual_output=actual_output,
        expected_output=golden.expected_output or "",
        retrieval_context=retrieval_ctx,
    )
    synthetic_test_cases.append(tc)

print(f"Created {len(synthetic_test_cases)} test cases from synthetic data.")

In [ ]:
# Evaluate synthetic test cases
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric

synth_metrics = [
    AnswerRelevancyMetric(threshold=0.7, model="gpt-4o-mini"),
    FaithfulnessMetric(threshold=0.7, model="gpt-4o-mini"),
]

if synthetic_test_cases:
    print(f"Evaluating {len(synthetic_test_cases)} synthetic test cases...\n")
    synth_results = deepeval_evaluate(
        test_cases=synthetic_test_cases,
        metrics=synth_metrics,
    )
    print("\nSynthetic data evaluation complete.")
else:
    print("No synthetic test cases to evaluate.")

---
## RAGAS TestsetGenerator

RAGAS provides its own test set generation approach. While DeepEval's Synthesizer generates individual test cases, RAGAS TestsetGenerator creates complete evaluation datasets with different query complexity levels.

-
## Part 1: Testset Generation

RAGAS can **automatically generate evaluation test sets** from your documents. This is incredibly valuable because:
- Manually creating golden test sets is expensive and time-consuming
- Generated test sets cover different difficulty levels and question types
- You can generate hundreds of test cases automatically

RAGAS TestsetGenerator creates questions with different **evolution types**:
- **Simple**: Direct factual questions that can be answered from a single chunk
- **Reasoning**: Questions requiring inference or multi-step reasoning
- **Multi-context**: Questions that require combining information from multiple chunks

In [ ]:
# Convert our documents to LangChain Document format for the generator
lc_documents = [
    LCDocument(
        page_content=f"{doc['title']}\n\n{doc['content']}",
        metadata={"title": doc["title"], "id": doc["id"]},
    )
    for doc in ACME_DOCUMENTS
]

print(f"Prepared {len(lc_documents)} LangChain documents for testset generation")

In [ ]:
# Initialize the TestsetGenerator
generator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
generator_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)

print("TestsetGenerator initialized.")

In [ ]:
# Generate a test set from our documents
# We request 10 test samples across different query types
testset = generator.generate_with_langchain_docs(
    documents=lc_documents,
    testset_size=10,
)

print(f"Generated {len(testset)} test samples")

In [ ]:
# Inspect the generated test cases
testset_df = testset.to_pandas()
print("Generated Test Cases:")
print("=" * 80)
for idx, row in testset_df.iterrows():
    print(f"\n--- Test Case {idx + 1} ---")
    print(f"  Question: {row.get('user_input', 'N/A')}")
    print(f"  Reference: {str(row.get('reference', 'N/A'))[:120]}...")
    print(f"  Contexts: {len(row.get('reference_contexts', []))} context(s)")

testset_df.head()

In [ ]:
# Now evaluate our RAG pipeline on the generated test set
# First, run each generated question through our RAG pipeline
generated_samples = []

for idx, row in testset_df.iterrows():
    question = row["user_input"]
    result = rag_pipeline(question)
    
    sample = SingleTurnSample(
        user_input=question,
        response=result["answer"],
        reference=str(row.get("reference", "")),
        retrieved_contexts=result["contexts"],
    )
    generated_samples.append(sample)
    print(f"Processed question {idx + 1}: {question[:60]}...")

generated_eval_dataset = RagasEvaluationDataset(samples=generated_samples)
print(f"\nCreated evaluation dataset with {len(generated_eval_dataset)} samples")

In [ ]:
# Evaluate with core metrics on the generated test set
evaluator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
evaluator_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

generated_results = ragas_evaluate(
    dataset=generated_eval_dataset,
    metrics=[
        Faithfulness(llm=evaluator_llm),
        ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
        LLMContextRecall(llm=evaluator_llm),
    ],
)

gen_results_df = generated_results.to_pandas()
print("Results on Generated Test Set:")
print("=" * 60)
print(f"  Faithfulness (mean):      {gen_results_df['faithfulness'].mean():.3f}")
print(f"  Answer Relevancy (mean):  {gen_results_df['answer_relevancy'].mean():.3f}")
print(f"  Context Recall (mean):    {gen_results_df['llm_context_recall'].mean():.3f}")
gen_results_df

---
## DeepEval Synthesizer vs RAGAS TestsetGenerator

| Feature | DeepEval Synthesizer | RAGAS TestsetGenerator |
|---------|---------------------|----------------------|
| Input | List of document strings | LangChain Documents |
| Output | List of Golden objects | Testset with DataFrames |
| Query types | Configurable scenarios | Simple, Multi-context, Reasoning |
| Reference answers | Generated automatically | Generated from documents |
| LLM requirement | Any DeepEval-compatible LLM | LangChain-wrapped LLM |
| Best for | Quick synthetic datasets | Comprehensive test coverage |

**Recommendation:** Use DeepEval Synthesizer for quick iteration during development, and RAGAS TestsetGenerator for comprehensive evaluation before releases.

---
## Summary

1. **EvaluationDataset** lets you manage test cases from JSON, CSV, or code
2. **Synthesizer** auto-generates test cases from your knowledge base documents
3. **Pytest integration** enables CI/CD evaluation pipelines
4. **RAGAS TestsetGenerator** provides complementary test generation

### Next Steps
- **Notebook 08**: Faithfulness deep dive — hallucination types, edge cases, mitigation strategies